# Study in Germany Assistant: run
Runtime → Change runtime type → **T4 GPU**. Start from an empty GPU
(Runtime → Disconnect and delete runtime), then run the cells from top to bottom, **once each**.

Runs entirely in Colab: the code and data come from GitHub, the model from Hugging Face (no Google Drive needed).
To use files on Google Drive instead, switch to the commented Option B in cells 3 and 5.

## 1. Setup

In [ ]:
# Option A (active): everything in Colab, no Google Drive (files are gone when the session ends)
![ -d /content/nlp-tasks-2026 ] || git clone -q https://github.com/cedricfyc/nlp-tasks-2026.git /content/nlp-tasks-2026
PROJECT = "/content/nlp-tasks-2026/Assignment 1"

# Option B: run from Google Drive (comment out Option A above, uncomment these 3 lines)
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT = "/content/drive/MyDrive/nlp-tasks-2026/Assignment 1"

%cd "$PROJECT/rag"

# docling is only needed to rebuild the database from the PDFs and is slow to install: skipped here
!grep -v docling ../requirements.txt > /tmp/requirements_run.txt
!pip install -q -r /tmp/requirements_run.txt

## 2. Model and Hugging Face login

In [ ]:
import os
from huggingface_hub import get_token, login, snapshot_download

# Option A (active): download Qwen3.5-4B from Hugging Face (~9 GB, about 3-10 min each session)
os.environ["RAG_MODEL_PATH"] = "/content/models/Qwen3.5-4B"
snapshot_download("Qwen/Qwen3.5-4B", local_dir=os.environ["RAG_MODEL_PATH"])

# Option B: model already on Google Drive (comment out the 2 lines above, uncomment this one)
# os.environ["RAG_MODEL_PATH"] = "/content/drive/MyDrive/models/Qwen3.5-4B"

# Hugging Face login, needed for SPLADE (accept the licence on huggingface.co/naver/splade-v3 first)
if get_token() is None:
    login()

## 3. Start the assistant (chat page + `POST /ask`)
Run once per session. A link to the chat page appears below the cell.

In [ ]:
from pipeline import RAGPipeline
from api import start_in_background

rag = RAGPipeline(use_splade=True,              # SPLADE as a third search (False: BM25 + dense only)
                  use_reranker=True,            # re-rank 20 candidates, keep the top k (False: off)
                  refusal_method="combined",    # 2-of-3 vote before answering ("none" | "bm25" | "dense" | "llm_judge")
                  retriever="both")             # BM25 + dense ("bm25" | "dense")
start_in_background(rag)

## 4. Ask from the notebook (optional)

In [ ]:
thinking, answer = rag.conduct_rag("What are the requirements for a blocked account in Germany?")
print(answer)